# AERO EYES — chạy trên vast.ai (nhánh `test-geco2`, detector GeCo2)

Notebook này: cài dependency → clone code (nhánh `test-geco2`) → clone dataset từ Google Drive → chạy pipeline (`pipeline.detector: geco2`) → gộp + đánh giá kết quả.

**Lưu ý riêng cho vast.ai (khác Colab/Kaggle):**
- Không có `google.colab.drive.mount()` hay `/kaggle/input` tự động — dataset phải tự tải bằng `gdown`.
- Chọn template vast.ai có sẵn **CUDA toolkit đầy đủ** (không chỉ runtime) — bước build extension CUDA bên dưới cần `nvcc`. Image kiểu `pytorch/pytorch:*-devel` hoặc template có ghi "CUDA" của vast.ai là an toàn nhất.
- Instance vast.ai có thể bị xoá bất cứ lúc nào sau khi bạn dừng thuê — **tải kết quả về trước khi kết thúc** (cell cuối cùng có hướng dẫn).

**Phát hiện quan trọng khi chuẩn bị notebook này:** thư mục `GECO2/` trong repo hiện đang được git track như một **gitlink rỗng** (không có `.gitmodules` đăng ký) — nghĩa là `git clone` bình thường **sẽ ra thư mục `GECO2/` trống**, không có code bên trong. Notebook này đã tự xử lý (clone bù trực tiếp từ upstream `jerpelhan/GECO2` đúng commit đã ghim), nhưng đây là một lỗ hổng hạ tầng nên fix triệt để trong repo — nhắc bạn chủ động sửa (đăng ký submodule đúng cách hoặc bỏ nested `.git` để track như file thường).

## 0. Biến cấu hình chung — chỉnh ở đây trước khi chạy

In [ ]:
import os

# --- Repo chính (branch test-geco2) ---
REPO_URL = "https://github.com/Hoang-hai-yen/Test.git"
REPO_BRANCH = "test-geco2"
REPO_DIR = "/workspace/Test"

# --- GECO2 upstream (bù cho gitlink rỗng, xem cảnh báo phía trên) ---
GECO2_UPSTREAM_URL = "https://github.com/jerpelhan/GECO2.git"
GECO2_PINNED_COMMIT = "5b4fe9c4bc4a453bb366d314b80efb81450c51ef"

# --- GeCo2 pretrained weights (public HuggingFace asset, xem GECO2/README.md) ---
GECO2_WEIGHTS_URL = "https://huggingface.co/datasets/jerpelhan/geco2-assets/resolve/main/weights/CNTQG_multitrain_ca44.pth?download=true"

# --- Dataset trên Google Drive: 1 file .zip ---
# Lấy FILE_ID từ link share dạng https://drive.google.com/file/d/<FILE_ID>/view
GDRIVE_FILE_ID = "PASTE_HERE"  # <-- ACTION REQUIRED: dán File ID thật vào đây

# --- Chạy sample nào? None/rỗng = chạy hết mọi sample trong data_root ---
SAMPLE_ID = ""  # ví dụ "BlackBox_0" để chạy 1 sample; để rỗng "" = chạy tất cả

# --- Detector: "geco2" (mục tiêu notebook này) hoặc "legacy" để so sánh ---
DETECTOR = "geco2"

os.environ.update({
    "REPO_URL": REPO_URL, "REPO_BRANCH": REPO_BRANCH, "REPO_DIR": REPO_DIR,
    "GECO2_UPSTREAM_URL": GECO2_UPSTREAM_URL, "GECO2_PINNED_COMMIT": GECO2_PINNED_COMMIT,
    "GECO2_WEIGHTS_URL": GECO2_WEIGHTS_URL, "GDRIVE_FILE_ID": GDRIVE_FILE_ID,
    "SAMPLE_ID": SAMPLE_ID, "DETECTOR": DETECTOR,
})
print("OK — nhớ điền GDRIVE_FILE_ID thật trước khi chạy cell tải dataset.")

## 1. Kiểm tra GPU / CUDA toolkit

In [ ]:
!nvidia-smi
!echo "--- nvcc (cần cho bước build CUDA extension ở dưới) ---"
!nvcc --version || echo "CẢNH BÁO: không thấy nvcc — chọn lại template vast.ai có CUDA toolkit (devel), không phải bản runtime-only."

## 1a. Đồng bộ python giữa Jupyter kernel và shell (`%%bash`)

**Lỗi thường gặp trên vast.ai:** `%%bash`/`!pip install` chạy trong 1 shell subprocess có thể trỏ tới **python khác** với chính Jupyter kernel đang chạy notebook này (kernel thường nằm trong 1 venv riêng, còn shell mặc định vào python hệ thống) — cài package qua `%%bash` xong nhưng `import` trong cell Python thuần vẫn báo `ModuleNotFoundError`.

Cell dưới ưu tiên đúng thư mục chứa `sys.executable` (python của kernel) lên đầu `PATH`, để mọi `%%bash pip install` / `%%bash python -m ...` sau đó dùng cùng 1 môi trường với kernel. **Chạy cell này SỚM NHẤT (trước mọi bước cài đặt)** — nếu bạn đã lỡ chạy các cell cài đặt trước đó rồi mới thêm cell này, hãy **Restart Kernel rồi Run All lại từ đầu** để đảm bảo mọi thứ cài đúng chỗ.

In [ ]:
import os
import shutil
import sys

kernel_python = sys.executable
kernel_bin = os.path.dirname(kernel_python)
shell_python3 = shutil.which("python3")

print("Kernel python (sys.executable):", kernel_python)
print("Shell python3 (which python3) trước khi sửa:", shell_python3)

if shell_python3 and os.path.realpath(shell_python3) != os.path.realpath(kernel_python):
    print(">>> LỆCH MÔI TRƯỜNG -- ưu tiên thư mục của kernel python lên đầu PATH.")
else:
    print(">>> Khớp nhau (hoặc không xác định được shell python3) -- vẫn set PATH cho chắc.")

os.environ["PATH"] = kernel_bin + os.pathsep + os.environ.get("PATH", "")

print("Shell python3 sau khi sửa:", shutil.which("python3"))
print("pip sau khi sửa:", shutil.which("pip"))

## 1b. Đảm bảo lệnh `python` tồn tại

Nhiều image vast.ai (Debian/Ubuntu) chỉ có `python3`, không có `python` -- mọi lệnh `python ...` trong notebook này (build extension, `run_all`, `evaluate`, ...) sẽ báo `command not found` nếu bỏ qua bước này.

In [ ]:
%%bash
set -e
if ! command -v python &> /dev/null; then
    PY3=$(command -v python3)
    echo "Không có lệnh 'python' -- tạo symlink tới $PY3"
    ln -sf "$PY3" /usr/local/bin/python
fi
python --version
python -m pip --version

## 2. Lấy code (clone lần đầu, pull các lần sau)

**An toàn để chạy lại nhiều lần:** nếu `$REPO_DIR` đã tồn tại (đã clone trước đó), cell này **`git pull`** thay vì xoá-clone-lại -- giữ nguyên `GECO2/models/ops/` (đã build ở mục 5) và `GECO2/CNTQG_multitrain_ca44.pth` (đã tải ở mục 7), không cần build/tải lại. Nếu chưa từng chạy, tự clone mới như bình thường.

Nếu `GECO2/` trống sau khi clone lần đầu (do gitlink chưa đăng ký — xem cảnh báo ở đầu notebook), tự clone bù từ upstream đúng commit đã ghim.

In [ ]:
%%bash
set -e
mkdir -p /workspace

if [ -d "$REPO_DIR/.git" ]; then
    echo ">>> $REPO_DIR đã tồn tại -- pull code mới (KHÔNG xoá GECO2/models/ops hay weights đã tải)."
    cd "$REPO_DIR"
    git fetch origin "$REPO_BRANCH"
    git checkout "$REPO_BRANCH"
    git reset --hard "origin/$REPO_BRANCH"
else
    echo ">>> Chưa có $REPO_DIR -- clone mới."
    git clone --branch "$REPO_BRANCH" "$REPO_URL" "$REPO_DIR"
    cd "$REPO_DIR"
fi

if [ -z "$(ls -A GECO2 2>/dev/null)" ]; then
    echo ">>> GECO2/ rỗng (gitlink chưa đăng ký) -- clone bù trực tiếp từ upstream ..."
    rm -rf GECO2
    git clone "$GECO2_UPSTREAM_URL" GECO2
    cd GECO2 && git checkout "$GECO2_PINNED_COMMIT" && cd ..
else
    echo ">>> GECO2/ đã có sẵn code -- giữ nguyên, không đụng vào."
fi

echo "--- kiểm tra ---"
ls "$REPO_DIR"
ls "$REPO_DIR/GECO2" | head -5

echo "--- trạng thái build/weights trong GECO2/ (không bị ảnh hưởng bởi git pull) ---"
[ -d "$REPO_DIR/GECO2/models/ops" ] && echo "models/ops: đã build (mục 5 không cần chạy lại)" \
    || echo "models/ops: CHƯA build -- cần chạy mục 5"
[ -f "$REPO_DIR/GECO2/CNTQG_multitrain_ca44.pth" ] && echo "weights: đã có (mục 7 không cần chạy lại)" \
    || echo "weights: CHƯA tải -- cần chạy mục 7"

## 3. Cài dependency của aero_eyes

In [ ]:
%%bash
set -e
cd "$REPO_DIR"
pip install -q -r requirements.txt
pip install -q -e .
# Cần cho stage4.tracker=builtin (CSRT/KCF) -- opencv-python thường KHÔNG có cv2.legacy
pip install -q opencv-contrib-python-headless
echo "Done: aero_eyes deps"

## 4. Cài dependency riêng của GECO2

Theo `GECO2/install.sh`, bỏ qua `gradio`/`gradio_image_prompter` (chỉ cần cho demo UI, pipeline không dùng).

**Không** `pip install` package `sam2` -- code GECO2 tự import theo kiểu namespace package hai lớp (`sam2.sam2.modeling...`) dựa vào việc `GECO2/` (không phải `GECO2/sam2/`) nằm trên `sys.path`, việc pip-install `sam2` như 1 package độc lập sẽ làm sai đường import này.

**Cũng bỏ luôn `huggingface-hub==0.34.3`** mà `install.sh` gốc pin -- đã kiểm tra: package đó trong `install.sh` chỉ phục vụ `gradio`/`SAM2ImagePredictor.from_pretrained()` (không dùng ở đây), còn code GeCo2 thật sự chạy (`models/counter_infer.py`, `models/sam_mask.py`, backbone) không import `huggingface_hub` ở đâu cả. Giữ pin này sẽ xung đột với `transformers` (aero_eyes cần bản `huggingface-hub>=1.5,<2.0`) -- bỏ đi là an toàn, không mất chức năng gì.

In [ ]:
%%bash
set -e
pip install -q hydra-core omegaconf iopath scikit-image pycocotools einops
# MobileSAM: cắt nền ảnh ref trước khi encode exemplar (stage123_geco2.segmentation.enabled=true,
# mặc định bật). Không có trên PyPI dưới tên "mobile-sam" -- cài thẳng từ repo gốc. "timm" là
# dependency của mobile_sam nhưng không tự kéo theo khi cài qua git -- cần cài riêng.
# --no-deps: setup.py của mobile_sam kéo theo torch/torchvision KHÔNG ghim version, sẽ ghi đè
# lên cặp torch/torchvision cu126 đã cài khớp nhau (vd ở cell fix compute_70) -- gây lệch ABI,
# hỏng "operator torchvision::nms does not exist" (MobileSAM load fail, fallback passthrough
# âm thầm). --no-deps giữ nguyên torch/torchvision hiện có, chỉ cài đúng mobile_sam.
# Có graceful fallback (passthrough, không cắt nền) nếu thiếu -- không bắt buộc để pipeline chạy được.
pip install -q timm
pip install -q --no-deps git+https://github.com/ChaoningZhang/MobileSAM.git
echo "--- kiểm tra torch/torchvision còn khớp nhau không ---"
python -c "import torch, torchvision; print('torch', torch.__version__, '| torchvision', torchvision.__version__); from torchvision.ops import nms; import torch as t; nms(t.tensor([[0.,0.,1.,1.]]), t.tensor([0.9]), 0.5); print('torchvision::nms OK')"
echo "Done: GECO2 deps (chưa pin numpy/pydantic -- xem cell pin version ở cuối)"

### (Khắc phục) torchvision lỗi `has no attribute 'extension'` / `operator torchvision::nms does not exist`

Chỉ chạy nếu mục 4 vẫn báo 1 trong 2 lỗi trên dù đã có `--no-deps`. Nguyên nhân: torch/torchvision trong venv đã bị cài đè qua nhiều lần khác nhau trong lúc debug (base image cu130 → fix compute_70 sang cu126 → có thể còn lần cài mobile_sam CHƯA `--no-deps` trước đó) -- state hiện tại không còn nhất quán, vá tiếp từng triệu chứng sẽ không hết. Cell dưới reinstall sạch cả 3 gói `torch`/`torchvision`/`torchaudio` làm 1 cặp khớp nhau (đè hoàn toàn lên state cũ), **không cần chạy lại apt CUDA toolkit** (phần đó không liên quan, vẫn giữ nguyên).

In [ ]:
%%bash
set -e
pip install -q --force-reinstall --no-deps \
    torch==2.7.1 torchvision==0.22.1 torchaudio==2.7.1 \
    --index-url https://download.pytorch.org/whl/cu126
echo "--- kiểm tra ---"
python -c "
import torch, torchvision
print('torch', torch.__version__, '| torchvision', torchvision.__version__, '| cuda available:', torch.cuda.is_available())
from torchvision.ops import nms
nms(torch.tensor([[0.,0.,1.,1.]]), torch.tensor([0.9]), 0.5)
print('torchvision::nms OK')
"

### (Tuỳ chọn) Fix `nvcc fatal: Unsupported gpu architecture 'compute_70'`

Chỉ chạy nếu bước build CUDA extension (mục 5) báo lỗi đúng như trên. Nguyên nhân: **`nvcc` của toolkit hệ thống trên image** đã lên CUDA 13.x và **bỏ hỗ trợ Volta/V100 (compute_70)**.

**ĐÃ XÁC NHẬN trên máy thật:** cài riêng `nvidia-cuda-nvcc-cu12` qua pip **không dùng được** — gói đó chỉ chứa `ptxas`/`nvvm` (thư viện backend), không có binary `nvcc` (compiler driver) thật. Cách đúng: cài hẳn **CUDA Toolkit 12.6 đầy đủ qua apt** (3 cell dưới: đặt cờ → apt install → hoàn tất env + đổi torch sang cu126). Tải khá nặng (~3-4GB), cần internet + quyền root trên instance (thường có sẵn trên vast.ai).

Nếu bước apt cũng lỗi (thiếu repo NVIDIA cho đúng bản OS, mạng chặn, v.v.): fallback chắc ăn nhất là **thuê lại instance vast.ai khác** với template ghi rõ CUDA ≤ 12.6 — đúng bản GECO2/install.sh đã test, không cần debug gì thêm.

In [ ]:
# Đặt True nếu build CUDA extension (mục 5) báo lỗi "nvcc fatal: Unsupported gpu architecture 'compute_70'".
FIX_NVCC_FOR_VOLTA = False

import os
import torch
print("torch hiện tại:", torch.__version__, "| cuda build:", torch.version.cuda,
      "| cuda available:", torch.cuda.is_available())

# Cell %%bash tiếp theo đọc cờ này qua biến môi trường.
os.environ["FIX_NVCC_FOR_VOLTA"] = "1" if FIX_NVCC_FOR_VOLTA else "0"

In [ ]:
%%bash
set -e
if [ "$FIX_NVCC_FOR_VOLTA" != "1" ]; then
    echo "FIX_NVCC_FOR_VOLTA=False -- bỏ qua cell này."
    exit 0
fi

. /etc/os-release
echo "OS phát hiện: $ID $VERSION_ID"
TAG="${ID}$(echo "$VERSION_ID" | tr -d '.')"
echo "Thử NVIDIA apt repo tag: $TAG"

KEYRING_URL="https://developer.download.nvidia.com/compute/cuda/repos/${TAG}/x86_64/cuda-keyring_1.1-1_all.deb"
if ! wget -q --spider "$KEYRING_URL"; then
    echo "Không có repo cho '$TAG', thử fallback 'ubuntu2404' ..."
    TAG="ubuntu2404"
    KEYRING_URL="https://developer.download.nvidia.com/compute/cuda/repos/${TAG}/x86_64/cuda-keyring_1.1-1_all.deb"
fi
echo "Dùng: $KEYRING_URL"

wget -q "$KEYRING_URL" -O /tmp/cuda-keyring.deb
dpkg -i /tmp/cuda-keyring.deb
apt-get update -qq
apt-get install -y -qq cuda-toolkit-12-6

echo "--- kiểm tra ---"
ls -d /usr/local/cuda-12.6
/usr/local/cuda-12.6/bin/nvcc --version

In [ ]:
if FIX_NVCC_FOR_VOLTA:
    import os, subprocess, sys

    cuda126_home = "/usr/local/cuda-12.6"
    nvcc126 = os.path.join(cuda126_home, "bin", "nvcc")
    if not os.path.exists(nvcc126):
        raise FileNotFoundError(
            f"{nvcc126} không tồn tại -- cell apt cài CUDA toolkit ở trên có thể đã lỗi, "
            "đọc lại log cell đó trước khi chạy tiếp."
        )

    os.environ["CUDA_HOME"] = cuda126_home
    os.environ["PATH"] = os.path.join(cuda126_home, "bin") + os.pathsep + os.environ["PATH"]

    # Torch cũng đổi sang build cu126 để khớp runtime lib với nvcc 12.6 vừa cài
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--root-user-action=ignore",
                     "torch==2.7.1", "torchvision==0.22.1", "torchaudio==2.7.1",
                     "--index-url", "https://download.pytorch.org/whl/cu126"], check=True)

    print("--- kiểm tra lại ---")
    print("CUDA_HOME =", os.environ["CUDA_HOME"])
    subprocess.run(["nvcc", "--version"], check=True)

## 5. Build CUDA extension (Deformable-DETR ops) — bắt buộc cho GeCo2

`query_generator.py` của GECO2 cần `models.ops.modules.ms_deform_attn.MSDeformAttn`. Bước này build extension CUDA rồi copy source wrapper vào đúng vị trí `GECO2/models/ops/` (giống hệt `GECO2/install.sh`).

In [ ]:
%%bash
set -e
cd "$REPO_DIR/GECO2/Deformable-DETR/models/ops"
# --no-build-isolation: setup.py cần "import torch" ngay lúc build (torch.utils.cpp_extension) --
# build isolation mặc định của pip tạo 1 venv tạm KHÔNG có torch, gây "ModuleNotFoundError: No module named 'torch'".
CUDA_VISIBLE_DEVICES=0 python -m pip install --no-build-isolation .
cd "$REPO_DIR/GECO2"
rm -rf ./models/ops
cp -r ./Deformable-DETR/models/ops ./models/ops
# import torch TRƯỚC khi import extension -- extension .so cần libc10.so (nằm trong torch/lib/),
# chỉ định vị được nếu torch đã load trong cùng process trước đó (code thật của pipeline luôn
# import torch sớm nên không gặp lỗi này -- đây chỉ là yêu cầu của lệnh kiểm tra 1 dòng này).
python -c "import torch; import MultiScaleDeformableAttention; print('MultiScaleDeformableAttention import OK')"
echo "Done: CUDA ops extension built + copied"

## 6. Pin version cuối cùng (chạy SAU CÙNG trong phần cài đặt)

`GECO2/install.sh` cố tình đặt 2 dòng này cuối cùng — pip cài các gói khác ở trên có thể kéo numpy 2.x / pydantic mới hơn lên, 2 lệnh này ép lại đúng version GECO2 cần.

In [ ]:
%%bash
set -e
pip install -q "numpy<2"
pip install -q --force-reinstall "pydantic<2.11"
python -c "import numpy, pydantic; print('numpy', numpy.__version__, '| pydantic', pydantic.VERSION)"

## 7. Tải trọng số GeCo2

In [ ]:
%%bash
set -e
cd "$REPO_DIR/GECO2"
wget -q --show-progress -O CNTQG_multitrain_ca44.pth "$GECO2_WEIGHTS_URL"
ls -lh CNTQG_multitrain_ca44.pth

## 8. Tải dataset từ Google Drive (.zip)

**ACTION REQUIRED:** điền `GDRIVE_FILE_ID` thật ở Cell 0 trước khi chạy cell này.

In [ ]:
%%bash
set -e
if [ "$GDRIVE_FILE_ID" = "PASTE_HERE" ]; then
    echo "CHƯA điền GDRIVE_FILE_ID thật ở Cell 0 -- bỏ qua bước tải dataset."
    echo "Các cell setup ở trên vẫn dùng được để test riêng; quay lại đây khi có File ID."
else
    pip install -q -U gdown
    mkdir -p /workspace/aero_eyes_dataset_raw
    # gdown >=4.x bỏ flag --id -- truyền thẳng ID/URL làm positional argument.
    gdown "$GDRIVE_FILE_ID" -O /workspace/aero_eyes_dataset.zip
    unzip -q -o /workspace/aero_eyes_dataset.zip -d /workspace/aero_eyes_dataset_raw
    echo "--- cấu trúc sau khi giải nén (kiểm tra kỹ trước khi set DATA_ROOT/GT_FILE ở dưới) ---"
    find /workspace/aero_eyes_dataset_raw -maxdepth 4
fi

## 9. Trỏ đúng đường dẫn dataset

**Xem output `find` ở cell trên rồi chỉnh `DATA_ROOT`/`GT_FILE` cho khớp** — Drive/zip có thể có thêm 1 lớp thư mục con tuỳ cách bạn nén (giống lưu ý đã gặp thật với Kaggle trong `docs/COLAB_KAGGLE_GUIDE.md`), đừng đoán.

In [ ]:
import os

# <-- CHỈNH theo output `find` ở cell trên
DATA_ROOT = "/workspace/aero_eyes_dataset_raw"
GT_FILE = f"{DATA_ROOT}/annotations (1).json"
WORK_DIR = "/workspace/runs/exp001"

os.environ.update({"DATA_ROOT": DATA_ROOT, "GT_FILE": GT_FILE, "WORK_DIR": WORK_DIR})

samples = [d for d in os.listdir(DATA_ROOT) if os.path.isdir(os.path.join(DATA_ROOT, d))] if os.path.isdir(DATA_ROOT) else []
print(f"Tìm thấy {len(samples)} sample(s):", samples[:10])
print("GT file tồn tại:", os.path.exists(GT_FILE))

## 10. Smoke test — chạy 1 sample trước

Chạy thử 1 sample (dùng `SAMPLE_ID` đặt ở Cell 0, hoặc sample đầu tiên tìm được) để bắt lỗi sớm trước khi chạy hết dataset.

In [ ]:
%%bash
set -e
cd "$REPO_DIR"

SMOKE_SAMPLE="${SAMPLE_ID}"
if [ -z "$SMOKE_SAMPLE" ]; then
    SMOKE_SAMPLE=$(ls "$DATA_ROOT" | head -1)
fi
echo "Smoke test sample: $SMOKE_SAMPLE"

python -m aero_eyes.stages.run_all \
    --config configs/config.yaml \
    --sample "$SMOKE_SAMPLE" \
    --set pipeline.detector="$DETECTOR" \
    --set data.data_root="$DATA_ROOT" \
    --set data.gt.global_file="$GT_FILE" \
    --set project.work_dir="$WORK_DIR"

## 11. Chạy toàn bộ pipeline

Chạy hết mọi sample nếu `SAMPLE_ID` (Cell 0) để rỗng, hoặc chỉ 1 sample nếu đã set. Nếu bị 0 detection hàng loạt (domain gap ground→aerial), hạ `stage123_geco2.score_threshold_ratio` (xem phần Troubleshooting cuối notebook).

In [ ]:
%%bash
set -e
cd "$REPO_DIR"

SAMPLE_ARG=""
if [ -n "$SAMPLE_ID" ]; then
    SAMPLE_ARG="--sample $SAMPLE_ID"
fi

python -m aero_eyes.stages.run_all \
    --config configs/config.yaml \
    $SAMPLE_ARG \
    --set pipeline.detector="$DETECTOR" \
    --set data.data_root="$DATA_ROOT" \
    --set data.gt.global_file="$GT_FILE" \
    --set project.work_dir="$WORK_DIR"

## 12. Tổng hợp kết quả

`run_all` đã tự gộp mọi `submission.json` thành `WORK_DIR/submission_all.json` (trừ khi chạy `--no-merge`). Cell dưới chạy lại bước gộp tường minh (an toàn, upsert theo `video_id`, không tạo trùng) rồi in tóm tắt.

In [ ]:
%%bash
set -e
cd "$REPO_DIR"
python -m scripts.merge_submissions \
    --config configs/config.yaml \
    --set data.data_root="$DATA_ROOT" \
    --set data.gt.global_file="$GT_FILE" \
    --set project.work_dir="$WORK_DIR"

In [ ]:
import json, os

merged_path = os.path.join(WORK_DIR, "submission_all.json")
data = json.load(open(merged_path, encoding="utf-8"))
print(f"submission_all.json: {len(data)} video(s)")
for e in data:
    n = len(e["annotations"][0]["bboxes"]) if e.get("annotations") else 0
    print(f"  {e['video_id']}: {n} frame(s) có box")

## 13. Đánh giá ST-IoU

In [ ]:
%%bash
set -e
cd "$REPO_DIR"
python -m aero_eyes.evaluate \
    --pred "$WORK_DIR/submission_all.json" \
    --gt "$GT_FILE" \
    --config configs/config.yaml

## 14. Đóng gói kết quả để tải về

vast.ai không tự lưu output như Kaggle's Output tab — **tải file này về trước khi dừng/huỷ instance**, qua file browser của Jupyter (chuột phải → Download) hoặc `scp -P <port> root@<host>:/workspace/aero_eyes_results.zip .` (lấy host/port từ nút Connect trên trang instance vast.ai).

In [ ]:
%%bash
set -e
cd /workspace
rm -f aero_eyes_results.zip
zip -q -r aero_eyes_results.zip \
    "$WORK_DIR/submission_all.json" \
    $(find "$WORK_DIR" -maxdepth 2 -name 'submission.json') \
    $(find "$WORK_DIR" -maxdepth 3 -path '*/viz/stage5/timeline.jpg')
ls -lh aero_eyes_results.zip
echo "Đã đóng gói: /workspace/aero_eyes_results.zip"

## Troubleshooting (đã gặp thật khi chạy dự án này trên các nền tảng khác, xem thêm `docs/COLAB_KAGGLE_GUIDE.md`)

| Lỗi | Nguyên nhân | Cách sửa |
|--|--|--|
| `bash: line N: python: command not found` | Image vast.ai chỉ có `python3`, không có `python` | Đã thêm Cell "1b" (tạo symlink) ngay sau bước kiểm tra GPU -- chạy cell đó trước mọi cell dùng lệnh `python` |
| `import torch` báo `ModuleNotFoundError` dù đã `pip install` xong | Kernel Jupyter chạy từ venv riêng (vd `/venv/main`), còn `%%bash`/`!pip` mặc định vào python hệ thống khác (`/usr/bin/python3`) — cài 1 nơi, kernel tìm 1 nơi khác | Đã thêm Cell "1a" (đồng bộ `PATH` theo `sys.executable`) ngay đầu notebook. Nếu gặp lỗi này: chạy cell "1a", rồi chạy LẠI toàn bộ cell cài đặt (mục 3, 4, 5) theo đúng thứ tự |
| `Getting requirements to build wheel ... ModuleNotFoundError: No module named 'torch'` (khi build ops, mục 5) | `pip install .` mặc định build trong venv cô lập tạm thời không có torch, trong khi `setup.py` cần `import torch` lúc build | Đã thêm `--no-build-isolation` vào lệnh `pip install .` ở mục 5 |
| `nvcc fatal: Unsupported gpu architecture 'compute_70'` (build ops, mục 5) | Toolkit CUDA hệ thống trên image là 13.x, đã bỏ hỗ trợ Volta/V100 (compute_70) — không phải lỗi version torch | Đặt `FIX_NVCC_FOR_VOLTA = True` ở 3 cell "Tuỳ chọn" trước mục 5 (cài hẳn CUDA Toolkit 12.6 qua apt — **không dùng cách pip `nvidia-cuda-nvcc-cu12`, đã xác nhận không hoạt động**), rồi chạy lại mục 5. Nếu apt cũng lỗi: thuê lại instance khác, chọn template CUDA ≤ 12.6 |
| `ImportError: libc10.so: cannot open shared object file` (ngay sau khi build ops thành công, mục 5) | Extension `.so` cần `libc10.so` (nằm trong `torch/lib/`), chỉ định vị được nếu `torch` đã import trong cùng process trước đó -- không phải lỗi build thật | Đã sửa lệnh kiểm tra ở mục 5 thành `python -c "import torch; import MultiScaleDeformableAttention; ..."` (import torch trước). Code pipeline thật không gặp lỗi này vì đã import torch từ sớm |
| `MobileSAM unavailable (No module named 'mobile_sam')` hoặc `'timm'` (mục "Stage 1"/GeCo2) | `mobile_sam` không có trên PyPI dưới tên đơn giản (cần cài từ GitHub); `timm` là dependency của nó nhưng không tự kéo theo khi cài qua git | Đã thêm `pip install timm` + `pip install --no-deps git+https://github.com/ChaoningZhang/MobileSAM.git` vào mục 4. Có graceful fallback (passthrough) nên KHÔNG làm pipeline lỗi nếu thiếu -- chỉ là cắt nền không có tác dụng |
| `MobileSAM unavailable (operator torchvision::nms does not exist)` hoặc `has no attribute 'extension' (circular import)` | torch/torchvision trong venv bị cài đè lẫn lộn qua nhiều lần khác nhau trong lúc debug (base image → fix compute_70 → có thể còn 1 lần cài mobile_sam chưa `--no-deps`) -- lệch ABI hoặc install nửa vời | Đã thêm `--no-deps` vào lệnh cài mobile_sam ở mục 4 (ngăn tái diễn). Nếu ĐÃ bị lỗi rồi: chạy cell "(Khắc phục) torchvision lỗi ..." ngay sau mục 4 -- reinstall sạch cả 3 gói torch/torchvision/torchaudio làm 1 cặp khớp nhau, không cần chạy lại apt CUDA toolkit |
| `GeCo2 checkpoint missing N params (random init)` | Bình thường nếu grouped-by-submodule chỉ ra `sam_mask` -- submodule refine mask bằng SAM2 mà `GeCo2Detector` không bao giờ gọi tới (chỉ dùng box+score); khả năng cao checkpoint được lưu lúc `training=True` (khi đó `sam_mask` chưa được tạo nên không có trong checkpoint) | **Đã xác nhận trên máy thật:** `missing=['sam_mask']`, `unexpected=['bbox_embed_aux', 'class_embed_aux']` (2 đầu chỉ dùng lúc training) -- đúng như dự đoán, vô hại. Nếu log của bạn liệt kê thứ khác (vd `backbone`, `adapt_features`) thì đó mới là vấn đề thật cần điều tra |
| Gần như MỌI keyframe đều ra ít nhất 1 detection (vd 681/681) dù video dài, vật thể chỉ xuất hiện 1 đoạn | `stage123_geco2.score_threshold_ratio` là ngưỡng TƯƠNG ĐỐI theo max của chính frame đó -- luôn giữ lại ít nhất 1 điểm mỗi frame, kể cả frame không có vật thể thật (GeCo2 train/eval trên FSC147, nơi ảnh nào cũng chắc chắn có vật thể) | Chạy `python -m scripts.check_geco2_score_separation --config configs/config.yaml --sample <id>` để kiểm chứng bằng dữ liệu thật -- nếu điểm "absent" và "present" không tách biệt, cần thêm ngưỡng tuyệt đối trước khi tin kết quả |
| `GECO2/` rỗng sau khi clone | gitlink chưa đăng ký `.gitmodules` trong repo | Notebook đã tự clone bù (Cell 2); nên fix triệt để trong repo sau |
| `nvcc: command not found` (không thấy nvcc luôn) | Template vast.ai không có CUDA toolkit nào cả (chỉ runtime) | Chọn lại instance/image có CUDA devel |
| `ModuleNotFoundError: MultiScaleDeformableAttention` | Bước 5 (build ops) chưa chạy hoặc chạy lỗi | Chạy lại toàn bộ Cell mục 5, đọc kỹ log lỗi build |
| `ImportError: sam2...` hoặc sai đường dẫn import | Lỡ `pip install` package `sam2` riêng (đừng làm việc này — xem ghi chú ở mục 4) | `pip uninstall sam2 SAM-2 -y` nếu đã trót cài |
| `transformers ... requires huggingface-hub<2.0,>=1.5.0, but you have huggingface-hub 0.34.3` | Cell mục 4 (bản cũ) pin `huggingface-hub==0.34.3` theo `install.sh` gốc — package đó GeCo2 không thật sự dùng, chỉ xung đột với `transformers` | Đã bỏ pin này khỏi Cell mục 4 hiện tại. Nếu đã lỡ chạy bản cũ: `pip install -q -U huggingface-hub` để trả lại bản `transformers` cần |
| `OpenCV tracker 'csrt' not found` (Stage 4) | Thiếu `cv2.legacy` | Đã cài `opencv-contrib-python-headless` ở mục 3 |
| Mọi sample đều 0 detection | GeCo2's score scale khác biệt theo video (domain gap ground→aerial) | Hạ `--set stage123_geco2.score_threshold_ratio=0.15` (mặc định 0.33) khi chạy lại Cell 11 |
| `data_root not found` | `DATA_ROOT` ở Cell 9 chưa khớp cấu trúc thật sau giải nén | Xem lại output `find` ở Cell 8, đừng đoán path |
| Mất hết kết quả sau khi đóng notebook | vast.ai xoá instance/volume khi bạn dừng thuê | Luôn chạy Cell 14 và tải `aero_eyes_results.zip` về trước khi dừng instance |

## Phụ lục — Thử nghiệm: dán exemplar vào CÙNG ảnh (giống cách demo HuggingFace)

Pipeline chính (`stage123_geco2.py`) dùng `encode_exemplars()`: encode 3 ảnh ref RIÊNG (backbone forward pass tách biệt), rồi cross-attend token đó vào ảnh query. Demo HuggingFace gốc của GeCo2 lại dùng đúng API thiết kế ban đầu: exemplar box nằm **trong cùng 1 ảnh** đang detect (roi_align thẳng trên feature của chính ảnh đó).

3 cell dưới test giả thuyết: dán 3 ảnh ref (đã cắt nền MobileSAM + resize nhỏ) lên 1 frame thật của video (có vật thể theo GT), rồi chạy GeCo2 theo đúng kiểu "exemplar cùng ảnh" -- xem có detect đúng vật thể thật trong frame đó tốt hơn cách hiện tại không.

**Không gọi thẳng `model(x, bboxes)` như demo gốc** -- checkpoint của chúng ta thiếu `sam_mask` (xem log trước đó), gọi thẳng sẽ đi qua bước sửa box bằng weight random, làm nhiễu so sánh. Cell dưới viết lại đúng phần logic *trước* `sam_mask` (roi_align exemplar + adapt_features + class/bbox head) -- y hệt cách `GeCo2Detector` hiện tại đang làm -- để phép so sánh chỉ khác đúng 1 biến: exemplar cùng ảnh hay khác ảnh.

Chạy sau khi đã có `detector`/`cfg` từ các cell setup phía trên (mục 0-9).

In [ ]:
import sys
sys.path.insert(0, f"{REPO_DIR}")
sys.path.insert(0, f"{REPO_DIR}/GECO2")  # để import models.*, utils.* kiểu bare y hệt geco2_detector.py

import os
os.chdir(REPO_DIR)  # đường dẫn tương đối trong config.yaml (./GECO2, configs/config.yaml, ...)
                     # giả định cwd = REPO_DIR như mọi cell %%bash (luôn "cd $REPO_DIR") -- cell
                     # Python thuần này thì chưa, cần tự đổi.

import cv2
import numpy as np
import torch
from pathlib import Path

from aero_eyes.config import load_config
from aero_eyes.models.geco2_detector import GeCo2Detector
from aero_eyes.models.segmentation import MobileSAMSegmenter
from aero_eyes.stages.stage123_geco2 import _apply_mask, _apply_ref_downscale, _load_ref_images
from aero_eyes.utils.io import load_gt
from aero_eyes.utils.video import read_frame

TEST_SAMPLE = SAMPLE_ID or "BlackBox_0"
PATCH_DOWNSCALE = 0.05  # khớp ref_downscale_factor bạn muốn thử
# True: dùng bbox KHÍT theo mask MobileSAM làm exemplar box cho GeCo2 (chỉ vùng vật thể thật,
#       không lẫn viền nền xám). False: dùng cả patch (giống encode_exemplars() hiện tại).
USE_TIGHT_MASK_BOX = True
# None = tự chọn frame giữa đoạn present dài nhất (mặc định cũ).
# Đặt số cụ thể (vd 34) để test đúng frame đó -- không cần sửa gì bên dưới.
FRAME_IDX_OVERRIDE = None
# Chỉ số (0-based, THEO THỨ TỰ FILE GỐC trong thư mục ref) các ảnh ref muốn LOẠI HẲN --
# không dán lên frame, không đưa vào GeCo2 để xây exemplar/prototype. Vd MobileSAM cắt sai
# ảnh ref_1 và bạn không muốn dùng ảnh đó nữa (dù nguyên vẹn hay đã mask) thì đặt [1].
# Rỗng = dùng đủ cả 3 ảnh như bình thường.
EXCLUDE_REF_INDICES: list[int] = []


def _mask_bbox(mask: np.ndarray):
    """Bbox khít quanh vùng True của mask. None nếu mask rỗng."""
    ys, xs = np.where(mask)
    if len(xs) == 0:
        return None
    return float(xs.min()), float(ys.min()), float(xs.max() + 1), float(ys.max() + 1)


cfg = load_config("configs/config.yaml", [
    "pipeline.detector=geco2",
    f"data.data_root={DATA_ROOT}",
    f"data.gt.global_file={GT_FILE}",
    f"project.work_dir={WORK_DIR}",
])

# --- 1. Chọn frame test: FRAME_IDX_OVERRIDE nếu có set, không thì tự chọn giữa đoạn present dài nhất ---
gt = load_gt(cfg.data.gt.global_file, TEST_SAMPLE)
present_frames = sorted(gt.keys())
if FRAME_IDX_OVERRIDE is not None:
    frame_idx = FRAME_IDX_OVERRIDE
    if frame_idx not in gt:
        print(f"CẢNH BÁO: frame {frame_idx} KHÔNG có trong GT (vật thể không xuất hiện ở frame này) "
              "-- không có GT box thật để vẽ/so sánh, chỉ dán patch + chạy detect thôi.")
else:
    frame_idx = present_frames[len(present_frames) // 2]
gt_box = gt.get(frame_idx)  # None nếu frame không có trong GT
if gt_box is not None:
    print(f"Frame test: {frame_idx} | GT box: ({gt_box.x1:.0f},{gt_box.y1:.0f})-({gt_box.x2:.0f},{gt_box.y2:.0f})")
else:
    print(f"Frame test: {frame_idx} | không có GT box")

data_root = Path(cfg.data.data_root)
video_files = list((data_root / TEST_SAMPLE).glob(cfg.data.video_glob))
frame_bgr = read_frame(video_files[0], frame_idx)
h_frame, w_frame = frame_bgr.shape[:2]
print("Frame shape:", frame_bgr.shape)

# --- 2. Load 3 ảnh ref, LOẠI ngay các ảnh trong EXCLUDE_REF_INDICES trước khi làm gì khác ---
ref_imgs_all = _load_ref_images(cfg, TEST_SAMPLE)
kept_indices = [i for i in range(len(ref_imgs_all)) if i not in EXCLUDE_REF_INDICES]
ref_imgs = [ref_imgs_all[i] for i in kept_indices]
if EXCLUDE_REF_INDICES:
    print(f"Đã loại ảnh ref (theo EXCLUDE_REF_INDICES): {EXCLUDE_REF_INDICES} "
          f"-- còn lại {len(ref_imgs)}/{len(ref_imgs_all)} ảnh (gốc index {kept_indices}).")

# --- 3. Cắt nền (MobileSAM) + resize nhỏ các ảnh ref còn lại ---
seg_cfg = cfg.stage123_geco2.segmentation
segmenter = MobileSAMSegmenter(weights_path=seg_cfg.weights, fallback_if_missing=seg_cfg.fallback_if_missing, min_area_frac=seg_cfg.min_area_frac, max_area_frac=seg_cfg.max_area_frac, score_ratio_floor=seg_cfg.score_ratio_floor)
masks = [segmenter.segment(img) for img in ref_imgs]
ref_masked = [_apply_mask(img, m) for img, m in zip(ref_imgs, masks)]

# Bbox khít quanh vật thể thật (theo mask), TRÊN ảnh ref GỐC (trước khi resize) --
# nếu mask rỗng/lỗi (fallback passthrough all-ones), coi cả ảnh là "vật thể".
ref_tight_boxes_orig = []
for orig_idx, m in zip(kept_indices, masks):
    b = _mask_bbox(m)
    ref_tight_boxes_orig.append(b if b is not None else (0.0, 0.0, float(m.shape[1]), float(m.shape[0])))
    is_full_frame = (b is None) or (b[0] <= 0 and b[1] <= 0 and b[2] >= m.shape[1] and b[3] >= m.shape[0])
    print(f"ref_{orig_idx}: MobileSAM mask bbox (ảnh gốc): {ref_tight_boxes_orig[-1]}"
          + ("  <-- CẢNH BÁO: gần như trọn khung hình, MobileSAM có thể không tách được nền" if is_full_frame else ""))

ref_small = [_apply_ref_downscale(img, PATCH_DOWNSCALE) for img in ref_masked]
for orig_idx, r in zip(kept_indices, ref_small):
    print(f"ref_{orig_idx} sau cắt nền + resize (factor={PATCH_DOWNSCALE}):", r.shape)

# --- 4. Dán các patch còn lại lên góc frame, tránh đè lên vùng GT ---
# Chọn góc xa GT box nhất trong 4 góc (nếu có GT box; không có thì mặc định top-left).
corners = {
    "top-left": (5, 5), "top-right": (w_frame - 5, 5),
    "bottom-left": (5, h_frame - 5), "bottom-right": (w_frame - 5, h_frame - 5),
}
if gt_box is not None:
    gt_cx, gt_cy = (gt_box.x1 + gt_box.x2) / 2, (gt_box.y1 + gt_box.y2) / 2
    corner_name = max(corners, key=lambda k: (corners[k][0] - gt_cx) ** 2 + (corners[k][1] - gt_cy) ** 2)
else:
    corner_name = "top-left"
print("Dán patch vào góc:", corner_name)

composite = frame_bgr.copy()
exemplar_boxes_px: list[list[float]] = []       # box = TOÀN patch đã dán
exemplar_tight_boxes_px: list[list[float]] = [] # box = KHÍT quanh vật thể (theo mask), trong toạ độ composite
cursor = list(corners[corner_name])
for patch, tight_orig, orig_img in zip(ref_small, ref_tight_boxes_orig, ref_imgs):
    ph, pw = patch.shape[:2]
    x1 = cursor[0] if "left" in corner_name else max(0, cursor[0] - pw)
    y1 = cursor[1] if "top" in corner_name else max(0, cursor[1] - ph)
    x2 = min(w_frame, x1 + pw)
    y2 = min(h_frame, y1 + ph)
    composite[y1:y2, x1:x2] = patch[: y2 - y1, : x2 - x1]
    exemplar_boxes_px.append([float(x1), float(y1), float(x2), float(y2)])

    # scale bbox khít từ toạ độ ảnh ref GỐC -> toạ độ patch đã downscale -> toạ độ composite
    oh, ow = orig_img.shape[:2]
    sx, sy = pw / ow, ph / oh  # tỉ lệ resize thật theo từng chiều (làm tròn có thể lệch nhẹ x/y)
    tx1, ty1, tx2, ty2 = tight_orig
    exemplar_tight_boxes_px.append([
        x1 + tx1 * sx, y1 + ty1 * sy, x1 + tx2 * sx, y1 + ty2 * sy,
    ])

    if "left" in corner_name:
        cursor[0] = x2 + 5
    else:
        cursor[0] = x1 - 5

print(f"Số exemplar thực tế dùng: {len(exemplar_boxes_px)} (đã loại {len(EXCLUDE_REF_INDICES)}/{len(ref_imgs_all)})")
print("Exemplar box = toàn patch (px, composite):", exemplar_boxes_px)
print("Exemplar box = khít theo mask (px, composite):", [[round(v, 1) for v in b] for b in exemplar_tight_boxes_px])
print("USE_TIGHT_MASK_BOX =", USE_TIGHT_MASK_BOX, "-- box này sẽ được đưa vào GeCo2 ở cell sau.")

preview = composite.copy()
for b in exemplar_boxes_px:
    cv2.rectangle(preview, (int(b[0]), int(b[1])), (int(b[2]), int(b[3])), (0, 255, 0), 1)   # xanh lá = toàn patch
for b in exemplar_tight_boxes_px:
    cv2.rectangle(preview, (int(b[0]), int(b[1])), (int(b[2]), int(b[3])), (0, 255, 255), 2) # vàng = khít theo mask
if gt_box is not None:
    cv2.rectangle(preview, (int(gt_box.x1), int(gt_box.y1)), (int(gt_box.x2), int(gt_box.y2)), (255, 0, 0), 2)  # xanh dương = GT thật
cv2.imwrite("/workspace/composite_test_frame.jpg", preview)
print("Đã lưu /workspace/composite_test_frame.jpg -- Xanh lá = toàn patch, Vàng = bbox khít theo mask MobileSAM"
      + (", Xanh dương = GT vật thể thật." if gt_box is not None else " (không có GT ở frame này)."))
print("Tải về xem trực quan trước khi chạy model -- nếu box vàng gần bằng box xanh lá,")
print("MobileSAM khả năng không tách được vật thể khỏi nền (xem cảnh báo phía trên nếu có).")

### (Chẩn đoán) Xem cả 3 mask ứng viên SAM đề xuất cho 1 ảnh ref cụ thể

`MobileSAMSegmenter.segment()` chọn mask **lớn nhất** trong khoảng diện tích hợp lệ (`min_area_frac`..`max_area_frac`), nhưng chỉ trong số các ứng viên SAM chấm điểm **trong khoảng `score_ratio_floor` so với điểm cao nhất** (mặc định 0.85 -- 3 điều chỉnh này nằm ở `stage123_geco2.segmentation` trong `config.yaml`, không cần sửa code). Nếu mask vẫn thiếu vật thể (bị cắt) -> hạ `score_ratio_floor`. Nếu mask lẫn quá nhiều nền ("cắt dư") -> tăng `score_ratio_floor` (về 1.0 = chỉ nhận đúng 1 ứng viên điểm cao nhất, y hệt hành vi cũ) hoặc hạ `max_area_frac`.

Cell dưới gọi thẳng SAM để lưu ảnh riêng cho **cả 3 ứng viên** (kèm score + % diện tích) -- xem ứng viên nào đúng hình dạng vật thể nhất, để biết tham số nào cần chỉnh.

In [ ]:
DIAG_REF_INDEX = 1  # đổi theo ảnh muốn xem (0-based, theo thứ tự file gốc trong thư mục ref)

diag_img = ref_imgs_all[DIAG_REF_INDEX]
dh, dw = diag_img.shape[:2]
segmenter._predictor.set_image(cv2.cvtColor(diag_img, cv2.COLOR_BGR2RGB))

margin = 0.05
box = np.array([dw * margin, dh * margin, dw * (1 - margin), dh * (1 - margin)])
masks3, scores3, _ = segmenter._predictor.predict(
    point_coords=np.array([[dw // 2, dh // 2]]),
    point_labels=np.array([1]),
    box=box,
    multimask_output=True,
)

cleaned3 = [segmenter._isolate_component_at_point(m, dw // 2, dh // 2) for m in masks3]
areas3 = [float(m.mean()) for m in cleaned3]  # area SAU KHI đã bỏ blob nền rời rạc (khớp segment() thật)
max_score3 = float(np.max(scores3))
confident_idx = [i for i, s in enumerate(scores3) if s >= max_score3 * segmenter.score_ratio_floor]
plausible_idx = [i for i in confident_idx
                 if segmenter.min_area_frac <= areas3[i] <= segmenter.max_area_frac]
picked_idx = max(plausible_idx, key=lambda i: areas3[i]) if plausible_idx else int(np.argmax(scores3))

print(f"Ảnh gốc: /workspace/sam_diag_ref{DIAG_REF_INDEX}_original.jpg")
cv2.imwrite(f"/workspace/sam_diag_ref{DIAG_REF_INDEX}_original.jpg", diag_img)

for i, (m_raw, m_clean, s, area) in enumerate(zip(masks3, cleaned3, scores3, areas3)):
    vis = diag_img.copy()
    overlay = np.zeros_like(vis)
    overlay[m_raw.astype(bool)] = (0, 0, 255)     # đỏ = phần mask RAW (trước khi bỏ blob rời rạc)
    overlay[m_clean] = (0, 255, 0)                # xanh lá = phần còn lại SAU khi bỏ blob rời rạc
    vis = cv2.addWeighted(vis, 0.6, overlay, 0.4, 0)
    path = f"/workspace/sam_diag_ref{DIAG_REF_INDEX}_candidate{i}.jpg"
    cv2.imwrite(path, vis)
    mark = "  <-- segment() sẽ chọn cái này" if i == picked_idx else ""
    dropped = m_raw.astype(bool).mean() - area
    extra = f"  (đã bỏ {dropped:.1%} diện tích blob rời rạc)" if dropped > 0.001 else ""
    print(f"Candidate {i}: score={s:.4f}  area={area:.1%}{extra}  -> {path}{mark}")

In [ ]:
from torchvision.ops import nms as torch_nms, roi_align
from utils.box_ops import boxes_with_scores  # GECO2/utils/box_ops.py
from utils.data import resize_and_pad  # GECO2/utils/data.py

_IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
_IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)


@torch.no_grad()
def forward_same_image(model, x, bboxes_px, device):
    """Y hệt CNT.forward() phần TRƯỚC sam_mask (roi_align exemplar +
    adapt_features + class/bbox head), nhưng exemplar roi_align thẳng trên
    ẢNH ĐANG DETECT (x) thay vì trên ảnh ref riêng -- đúng API gốc GeCo2.
    bboxes_px: tensor [1,K,4] toạ độ pixel TUYỆT ĐỐI trên canvas ĐÃ PAD.
    """
    m = model
    num_objects = bboxes_px.size(1)
    feats = m.backbone(x)
    src = feats["vision_features"]
    l1 = feats["backbone_fpn"][0]
    l2 = feats["backbone_fpn"][1]
    bs, c, w, h = src.shape
    reduction = x.shape[-1] / w

    batch_idx = torch.zeros((bs * num_objects, 1), device=device)
    boxes_flat = bboxes_px.reshape(-1, 4).to(device)
    bboxes_roi = torch.cat([batch_idx, boxes_flat], dim=1)

    exemplars = roi_align(src, boxes=bboxes_roi, output_size=1, spatial_scale=1.0 / reduction, aligned=True)
    exemplars = exemplars.permute(0, 2, 3, 1).reshape(bs, num_objects, m.emb_dim)
    exemplars_l1 = roi_align(l1, boxes=bboxes_roi, output_size=1, spatial_scale=1.0 / reduction * 2 * 2, aligned=True)
    exemplars_l1 = exemplars_l1.permute(0, 2, 3, 1).reshape(bs, num_objects, m.emb_dim)
    exemplars_l2 = roi_align(l2, boxes=bboxes_roi, output_size=1, spatial_scale=1.0 / reduction * 2, aligned=True)
    exemplars_l2 = exemplars_l2.permute(0, 2, 3, 1).reshape(bs, num_objects, m.emb_dim)

    box_hw = torch.zeros(bs, num_objects, 2, device=device)
    box_hw[:, :, 0] = bboxes_px[:, :, 2] - bboxes_px[:, :, 0]
    box_hw[:, :, 1] = bboxes_px[:, :, 3] - bboxes_px[:, :, 1]
    shape = m.shape_or_objectness(box_hw).reshape(bs, num_objects, m.emb_dim)

    prototype_embeddings = torch.cat([exemplars, shape], dim=1)
    prototype_embeddings_l1 = torch.cat([exemplars_l1, shape], dim=1)
    prototype_embeddings_l2 = torch.cat([exemplars_l2, shape], dim=1)

    adapted_f, _ = m.adapt_features(
        image_embeddings=src, image_pe=m.sam_prompt_encoder.get_dense_pe(),
        prototype_embeddings=prototype_embeddings, hq_features=feats["backbone_fpn"],
        hq_prototypes=[prototype_embeddings_l1, prototype_embeddings_l2], hq_pos=feats["vision_pos_enc"],
    )
    bs2, c2, w2, h2 = adapted_f.shape
    adapted_f = adapted_f.view(bs2, m.emb_dim, -1).permute(0, 2, 1)
    centerness = m.class_embed(adapted_f).view(bs2, w2, h2, 1).permute(0, 3, 1, 2)
    outputs_coord = m.bbox_embed(adapted_f).sigmoid().view(bs2, w2, h2, 4).permute(0, 3, 1, 2)
    outputs, _ = boxes_with_scores(centerness, outputs_coord, sort=False, validate=True)
    return outputs[0]["pred_boxes"], outputs[0]["box_v"]


# --- 4. Chạy GeCo2 với exemplar CÙNG ẢNH ---
detector = GeCo2Detector(cfg)
model, device = detector.model, detector.device

img_rgb = cv2.cvtColor(composite, cv2.COLOR_BGR2RGB)
img_t = torch.from_numpy(img_rgb).permute(2, 0, 1).float() / 255.0
img_t = (img_t - _IMAGENET_MEAN) / _IMAGENET_STD

# USE_TIGHT_MASK_BOX (đặt ở cell trước): dùng bbox khít theo mask MobileSAM, hay cả patch (viền xám).
boxes_for_model = exemplar_tight_boxes_px if USE_TIGHT_MASK_BOX else exemplar_boxes_px
boxes_t = torch.tensor(boxes_for_model, dtype=torch.float32)  # [K,4], toạ độ trên ẢNH GỐC (composite)
padded_img, padded_boxes, scale = resize_and_pad(img_t, boxes_t, size=float(cfg.stage123_geco2.image_size), zero_shot=True)

x = padded_img.unsqueeze(0).to(device)
bboxes_padded = padded_boxes.unsqueeze(0).to(device)  # [1,K,4], toạ độ trên canvas ĐÃ PAD

pred_boxes, box_v = forward_same_image(model, x, bboxes_padded, device)
if box_v.numel() == 0:
    print("Không có box nào.")
else:
    print(f"So luong box tho: {pred_boxes.shape[0]} | max score: {float(box_v.max()):.4f}")

# --- 5. Threshold + NMS (dùng đúng config hiện tại) ---
ratio = cfg.stage123_geco2.score_threshold_ratio
result_boxes, result_scores = [], []
if box_v.numel() > 0:
    keep_mask = box_v > (box_v.max() * ratio)
    cand_boxes = torch.clamp(pred_boxes[keep_mask], 0, 1)
    cand_scores = box_v[keep_mask]
    keep_idx = torch_nms(cand_boxes, cand_scores, cfg.stage123_geco2.nms_iou)
    cand_boxes = cand_boxes[keep_idx]
    cand_scores = cand_scores[keep_idx]
    px_boxes = (cand_boxes / scale * cfg.stage123_geco2.image_size).cpu().numpy()
    result_scores = cand_scores.cpu().numpy().tolist()
    result_boxes = px_boxes.tolist()

# --- 6. Log RIÊNG toạ độ box vật thể GeCo2 detect được (bỏ qua box của 3 ảnh mẫu) ---
print(f"\n=== Bbox object detect được (frame {frame_idx}, sau threshold ratio={ratio} + NMS): "
      f"{len(result_boxes)} box ===")
for (bx1, by1, bx2, by2), s in sorted(zip(result_boxes, result_scores), key=lambda t: -t[1]):
    print(f"  ({bx1:.1f}, {by1:.1f}) - ({bx2:.1f}, {by2:.1f})  score={s:.4f}")
if gt_box is not None:
    print(f"  [so sánh] GT thật: ({gt_box.x1:.1f}, {gt_box.y1:.1f}) - ({gt_box.x2:.1f}, {gt_box.y2:.1f})")

# --- 7. Vẽ kết quả ---
vis = composite.copy()
for (bx1, by1, bx2, by2), s in zip(result_boxes, result_scores):
    cv2.rectangle(vis, (int(bx1), int(by1)), (int(bx2), int(by2)), (0, 0, 255), 2)  # đỏ = detect
    cv2.putText(vis, f"{s:.2f}", (int(bx1), max(0, int(by1) - 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
for b in exemplar_boxes_px:
    cv2.rectangle(vis, (int(b[0]), int(b[1])), (int(b[2]), int(b[3])), (0, 255, 0), 1)  # xanh lá = toàn patch
for b in exemplar_tight_boxes_px:
    cv2.rectangle(vis, (int(b[0]), int(b[1])), (int(b[2]), int(b[3])), (0, 255, 255), 2)  # vàng = khít theo mask (box thật đưa vào model nếu USE_TIGHT_MASK_BOX)
if gt_box is not None:
    cv2.rectangle(vis, (int(gt_box.x1), int(gt_box.y1)), (int(gt_box.x2), int(gt_box.y2)), (255, 0, 0), 2)  # xanh dương = GT thật

cv2.imwrite("/workspace/composite_test_result.jpg", vis)
print("\nĐã lưu /workspace/composite_test_result.jpg")
print("  Đỏ = box GeCo2 detect | Xanh lá = toàn patch | Vàng = bbox khít theo mask" + (" | Xanh dương = GT vật thể thật" if gt_box is not None else " | (không có GT ở frame này)"))
print(f"  Box thực tế đưa vào model làm exemplar: {'VÀNG (khít theo mask)' if USE_TIGHT_MASK_BOX else 'XANH LÁ (toàn patch)'}")
if gt_box is not None:
    print("  Kỳ vọng: có 1 box đỏ trùng khớp box xanh dương (detect đúng vật thể thật),")
    print("  không nhất thiết phải detect lại chính patch (đó chỉ là nguồn exemplar).")
else:
    print("  Frame này không có GT -- tự xem box đỏ (nếu có) có rơi đúng vị trí vật thể thật trong ảnh không.")